# Clean Questionnaire Data Pre/Post

In [11]:
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np
import pingouin as pg

import matplotlib.pyplot as plt
import seaborn as sns

from fau_colors import cmaps
import biopsykit as bp

%load_ext autoreload
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
plt.close("all")

palette = sns.color_palette(cmaps.faculties)
sns.set_theme(context="notebook", style="ticks", palette=palette)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["mathtext.default"] = "regular"

palette

[(0.0, 0.1843137254901961, 0.4235294117647059),
 (0.4666666666666667, 0.6235294117647059, 0.7098039215686275),
 (1.0, 0.7215686274509804, 0.10980392156862745),
 (0.0, 0.6392156862745098, 0.8784313725490196),
 (0.2627450980392157, 0.6901960784313725, 0.16470588235294117),
 (0.7843137254901961, 0.06274509803921569, 0.1803921568627451)]

In [13]:
deploy_type = "local"

In [33]:
config_dict = json.load(Path("../config.json").open(encoding="utf-8"))

base_path = Path(config_dict[deploy_type]["base_path"])
base_path

PosixPath('/Volumes/luca_ssd/Study_Data/2022_12_GAPVII_Micro')

In [15]:
code_mapping = pd.read_csv(base_path.joinpath("data_tabular/_extras/processed/id_mapping.csv"))
code_mapping = code_mapping.set_index("Code")
code_mapping.head()

,subject
Code,
LI17P,VP_001
IH30R,VP_002
HB25M,VP_003
SK29B,VP_004
BK24I,VP_005


## Load Pre Questionnaire Data and Map Code to VP-ID

In [16]:
quest_data = pd.read_excel(base_path.joinpath("data_tabular/questionnaires/processed/unipark_pre.xlsx"))
quest_data = quest_data.rename(columns={"VPN_Kennung": "Code", "Tag": "day"})
quest_data = quest_data.set_index("Code")

quest_data = quest_data.join(code_mapping)

# add T before day column
quest_data["day"] = "T" + quest_data["day"].astype(str)

quest_data = quest_data.dropna(subset=["subject"]).set_index(["subject", "day"])

quest_data = quest_data.filter(like="Pre")

# put pre to the end

quest_data.columns = quest_data.columns.str.replace("Pre_", "")

quest_data.columns = quest_data.columns + "_pre"

quest_data

STADI_1_pre  STADI_2_pre  STADI_3_pre  STADI_4_pre  STADI_5_pre  \
subject day                                                                    
VP_074  T1             2            1            3            1            1   
        T2             1            2            3            1            1   
VP_009  T1             1            1            3            2            1   
        T2             1            1            3            1            1   
VP_033  T1             2            1            3            1            1   
...                  ...          ...          ...          ...          ...   
VP_014  T2             1            1            2            1            2   
VP_071  T1             1            3            3            2            1   
        T2             1            1            3            1            1   
VP_030  T1             1            1            3            1            1   
        T2             2            2            3            1            2   

             STADI_6_pre  STADI_7_pre  STADI_8_pre  STADI_9_pre  STADI_10_pre  \
subject day                                                                     
VP_074  T1             2            3            1            2             3   
        T2             2            3            1            1             3   
VP_009  T1             3            4            1            1             1   
        T2             2            4            2            1             2   
VP_033  T1             2            3            1            1             2   
...                  ...          ...          ...          ...           ...   
VP_014  T2             2            2            1            1             2   
VP_071  T1             2            3            1            2             2   
        T2             2            3            1            1             1   
VP_030  T1             1            4            1            1             1   
        T2             2            4            1            1             2   

             ...  SSSQ_15_pre  SSSQ_16_pre  SSSQ_17_pre  SSSQ_18_pre  \
subject day  ...                                                       
VP_074  T1   ...            4            3            2            2   
        T2   ...            4            2            2            4   
VP_009  T1   ...            5            1            5            2   
        T2   ...            5            1            5            1   
VP_033  T1   ...            5            3            4            2   
...          ...          ...          ...          ...          ...   
VP_014  T2   ...            4            1            1            2   
VP_071  T1   ...            4            2            5            2   
        T2   ...            3            1            4            2   
VP_030  T1   ...            3            1            4            1   
        T2   ...            4            1            3            1   

             SSSQ_19_pre  SSSQ_20_pre  SSSQ_21_pre  SSSQ_22_pre  SSSQ_23_pre  \
subject day                                                                    
VP_074  T1             2            4            3            2            2   
        T2             4            4            2            3            4   
VP_009  T1             2            3            5            5            1   
        T2             2            2            4            4            1   
VP_033  T1             2            2            3            3            1   
...                  ...          ...          ...          ...          ...   
VP_014  T2             1            1            3            2            1   
VP_071  T1             2            2            4            4            2   
        T2             2            2            4            3            4   
VP_030  T1             2            2            4            4            3   
        T2           

## Load Post Questionnaire Data and Join With Pre

In [17]:
quest_data_post = pd.read_excel(base_path.joinpath("data_tabular/questionnaires/processed/unipark_post.xlsx"))
quest_data_post = quest_data_post.rename(columns={"VPN_Kennung": "Code", "Tag": "day"})
quest_data_post = quest_data_post.set_index("Code")

quest_data_post = quest_data_post.join(code_mapping)

# add T before day column
quest_data_post["day"] = "T" + quest_data_post["day"].astype(str)

quest_data_post = quest_data_post.dropna(subset=["subject"]).set_index(["subject", "day"])

quest_data_post = quest_data_post.filter(like="Post")

# put post to the end

quest_data_post.columns = quest_data_post.columns.str.replace("Post_", "")

quest_data_post.columns = quest_data_post.columns + "_post"

quest_data_post

STADI_1_post  STADI_2_post  STADI_3_post  STADI_4_post  \
subject day                                                           
VP_074  T1              1             3             2             1   
        T2              3             3             1             2   
VP_009  T1              2             1             2             3   
        T2              2             2             4             2   
VP_033  T1              2             1             3             1   
...                   ...           ...           ...           ...   
VP_014  T2              2             1             1             1   
VP_071  T2              1             1             4             1   
        T1              1             2             3             1   
VP_030  T1              2             2             4             1   
        T2              2             2             4             1   

             STADI_5_post  STADI_6_post  STADI_7_post  STADI_8_post  \
subject day                                                           
VP_074  T1              1             3             3             1   
        T2              2             3             2             2   
VP_009  T1              1             1             2             3   
        T2              1             2             4             1   
VP_033  T1              3             1             3             1   
...                   ...           ...           ...           ...   
VP_014  T2              1             2             1             1   
VP_071  T2              1             1             4             1   
        T1              1             1             3             1   
VP_030  T1              2             1             4             1   
        T2              2             3             3             1   

             STADI_9_post  STADI_10_post  ...  SSGS_6_post  SSGS_7_post  \
subject day                               ...                             
VP_074  T1              1              3  ...          4.0          3.0   
        T2              3              3  ...          5.0          1.0   
VP_009  T1              2              3  ...          4.0          4.0   
        T2              1              2  ...          4.0          4.0   
VP_033  T1              1              1  ...          1.0          4.0   
...                   ...            ...  ...          ...          ...   
VP_014  T2              2              1  ...          4.0          1.0   
VP_071  T2              1              1  ...          1.0          4.0   
        T1              1              1  ...          NaN          NaN   
VP_030  T1              1              2  ...          2.0          3.0   
        T2              3              3  ...          4.0          5.0   

             SSGS_8_post  SSGS_9_post  SSGS_10_post  SSGS_11_post  \
subject day                                                         
VP_074  T1           2.0          2.0           3.0           2.0   
        T2           4.0          4.0           1.0           4.0   
VP_009  T1           1.0          1.0           4.0           1.0   
        T2           1.0          1.0           4.0           1.0   
VP_033  T1           1.0          1.0           4.0           1.0   
...                  ...          ...           ...           ...   
VP_014  T2           1.0          1.0           1.0           1.0   
VP_071  T2           1.0          1.0           3.0           1.0   
        T1           NaN          NaN           NaN           NaN   
VP_030  T1           2.0          2.0           3.0           2.0   
        T2           3.0          1.0           2.0           3.0   

             SSGS_12_post  SSGS_13_post  SSGS_14_post  SSGS_15_post  
subject day                                                          
VP_074  T1            3.0           3.0           2.0           2.0  
        T2            4.0           1.0           4.0           4.0  
VP_009  T1 

In [18]:
quest_data_all = quest_data.join(quest_data_post)

In [19]:
quest_data_all.replace(-77, np.nan, inplace=True)

In [20]:
quest_data_all

STADI_1_pre  STADI_2_pre  STADI_3_pre  STADI_4_pre  STADI_5_pre  \
subject day                                                                    
VP_074  T1             2            1            3            1            1   
        T2             1            2            3            1            1   
VP_009  T1             1            1            3            2            1   
        T2             1            1            3            1            1   
VP_033  T1             2            1            3            1            1   
...                  ...          ...          ...          ...          ...   
VP_014  T2             1            1            2            1            2   
VP_071  T1             1            3            3            2            1   
        T2             1            1            3            1            1   
VP_030  T1             1            1            3            1            1   
        T2             2            2            3            1            2   

             STADI_6_pre  STADI_7_pre  STADI_8_pre  STADI_9_pre  STADI_10_pre  \
subject day                                                                     
VP_074  T1             2            3            1            2             3   
        T2             2            3            1            1             3   
VP_009  T1             3            4            1            1             1   
        T2             2            4            2            1             2   
VP_033  T1             2            3            1            1             2   
...                  ...          ...          ...          ...           ...   
VP_014  T2             2            2            1            1             2   
VP_071  T1             2            3            1            2             2   
        T2             2            3            1            1             1   
VP_030  T1             1            4            1            1             1   
        T2             2            4            1            1             2   

             ...  SSGS_6_post  SSGS_7_post  SSGS_8_post  SSGS_9_post  \
subject day  ...                                                       
VP_074  T1   ...          4.0          3.0          2.0          2.0   
        T2   ...          5.0          1.0          4.0          4.0   
VP_009  T1   ...          4.0          4.0          1.0          1.0   
        T2   ...          4.0          4.0          1.0          1.0   
VP_033  T1   ...          1.0          4.0          1.0          1.0   
...          ...          ...          ...          ...          ...   
VP_014  T2   ...          4.0          1.0          1.0          1.0   
VP_071  T1   ...          NaN          NaN          NaN          NaN   
        T2   ...          1.0          4.0          1.0          1.0   
VP_030  T1   ...          2.0          3.0          2.0          2.0   
        T2   ...          4.0          5.0          3.0          1.0   

             SSGS_10_post  SSGS_11_post  SSGS_12_post  SSGS_13_post  \
subject day                                                           
VP_074  T1            3.0           2.0           3.0           3.0   
        T2            1.0           4.0           4.0           1.0   
VP_009  T1            4.0           1.0           1.0           4.0   
        T2            4.0           1.0           1.0           3.0   
VP_033  T1            4.0           1.0           1.0           3.0   
...                   ...           ...           ...           ...   
VP_014  T2            1.0           1.0           2.0           1.0   
VP_071  T1            NaN           NaN           NaN           NaN   
        T2            3.0           1.0           1.0           4.0   
VP_030  T1            3.0           2.0           4.0           2.0   
        T2            2.0           3.0           3.0           3.0   

             SSGS_14_post  SSGS_15_post  
subject day    

In [21]:
# map day to condition

day_to_condition = pd.read_csv(base_path.joinpath("data_tabular/_extras/processed/day_condition_mapping.csv"))


In [22]:
day_to_condition.set_index(["subject", "day"], inplace=True)


In [23]:
quest_data_all = quest_data_all.join(day_to_condition).reset_index()
quest_data_all.set_index(["subject", "condition"], inplace=True)
quest_data_all.drop(columns="day", inplace=True)

In [24]:
# add trailing zeros

column_names_split = quest_data_all.columns.str.split("_")

quest_data_all.columns = column_names_split.str[0] + "_" + column_names_split.str[1].str.zfill(2) + "_" + column_names_split.str[2]

In [25]:
quest_data_all.sort_index(inplace=True)

In [26]:
# unstack condition
quest_data_all = quest_data_all.unstack("condition")

In [27]:
quest_data_all.sort_index(axis=1, level=1, inplace=True)

In [28]:
# squeeze multiindex
quest_data_all.columns = ["_".join(a) for a in quest_data_all.columns.to_flat_index()]

In [29]:
quest_data_sorted = quest_data_all.filter(like="pre").join(quest_data_all.filter(like="post"))

In [31]:
quest_data_sorted.filter(like="PANAS")

,PANAS_01_pre_ftsst,PANAS_02_pre_ftsst,PANAS_03_pre_ftsst,PANAS_04_pre_ftsst,PANAS_05_pre_ftsst,PANAS_06_pre_ftsst,PANAS_07_pre_ftsst,PANAS_08_pre_ftsst,PANAS_09_pre_ftsst,PANAS_10_pre_ftsst,...,PANAS_11_post_tsst,PANAS_12_post_tsst,PANAS_13_post_tsst,PANAS_14_post_tsst,PANAS_15_post_tsst,PANAS_16_post_tsst,PANAS_17_post_tsst,PANAS_18_post_tsst,PANAS_19_post_tsst,PANAS_20_post_tsst
subject,,,,,,,,,,,,,,,,,,,,,
VP_001,3.0,1.0,3.0,2.0,1.0,2.0,1.0,1.0,1.0,2.0,...,1.0,2.0,1.0,1.0,4.0,3.0,2.0,4.0,2.0,1.0
VP_002,2.0,2.0,4.0,4.0,2.0,4.0,1.0,1.0,1.0,3.0,...,3.0,1.0,3.0,1.0,4.0,2.0,3.0,3.0,1.0,1.0
VP_003,2.0,2.0,2.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,3.0,3.0
VP_004,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,1.0,1.0
VP_005,2.0,1.0,2.0,2.0,1.0,3.0,1.0,2.0,1.0,2.0,...,4.0,1.0,2.0,3.0,4.0,4.0,2.0,4.0,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VP_106,3.0,2.0,4.0,3.0,1.0,2.0,1.0,1.0,1.0,2.0,...,1.0,1.0,2.0,1.0,3.0,4.0,2.0,3.0,5.0,1.0
VP_107,3.0,1.0,5.0,3.0,1.0,1.0,2.0,1.0,1.0,3.0,...,1.0,4.0,1.0,2.0,3.0,2.0,4.0,3.0,2.0,2.0
VP_108,4.0,2.0,3.0,3.0,1.0,3.0,1.0,1.0,1.0,3.0,...,3.0,1.0,2.0,2.0,4.0,3.0,3.0,4.0,2.0,1.0


## PANAS

In [35]:
quest_data_sorted.filter(like="PANAS").to_csv(base_path.joinpath("data_tabular/questionnaires/final/panas.csv"))

## SSSQ

In [46]:
quest_data_sorted.filter(like="SSSQ").to_csv(base_path.joinpath("data_tabular/questionnaires/final/sssq.csv"))

## STADI

In [47]:
quest_data_sorted.filter(like="STADI").to_csv(base_path.joinpath("data_tabular/questionnaires/final/stadi.csv"))

## SSGS

In [48]:
quest_data_sorted.filter(like="SSGS").to_csv(base_path.joinpath("data_tabular/questionnaires/final/ssgs.csv"))